In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

# 1. LOCALIZAR RAÍZ DEL PROYECTO

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = None

for candidate in [CURRENT_DIR, *CURRENT_DIR.parents]:

    expected_dataset = (
        candidate
        / "data"
        / "processed"
        / "MMOTU_OTU_2D"
    )

    if expected_dataset.exists():
        PROJECT_ROOT = candidate
        break


if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se pudo encontrar la raíz del proyecto "
        "que contiene data/processed/MMOTU_OTU_2D."
    )


# 2. RUTAS DEL DATASET

DATASET_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "MMOTU_OTU_2D"
)

TRAIN_IMAGES_DIR = DATASET_DIR / "train" / "images"
TRAIN_MASKS_DIR = DATASET_DIR / "train" / "masks"

VAL_IMAGES_DIR = DATASET_DIR / "validation" / "images"
VAL_MASKS_DIR = DATASET_DIR / "validation" / "masks"

METADATA_PATH = DATASET_DIR / "metadata.csv"
EXCLUSIONS_PATH = DATASET_DIR / "exclusions.csv"


IMAGE_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff",
}

# 3. VERIFICACIÓN

print("Directorio actual:")
print(CURRENT_DIR)

print("\nRaíz del proyecto:")
print(PROJECT_ROOT)

print("\nDataset:")
print(DATASET_DIR)

print("\nVerificación de rutas:")

paths = {
    "Train images": TRAIN_IMAGES_DIR,
    "Train masks": TRAIN_MASKS_DIR,
    "Validation images": VAL_IMAGES_DIR,
    "Validation masks": VAL_MASKS_DIR,
    "Metadata": METADATA_PATH,
    "Exclusions": EXCLUSIONS_PATH,
}

for name, path in paths.items():
    status = "OK" if path.exists() else "NO ENCONTRADO"
    print(f"{name:<20} {status:<15} {path}")

Directorio actual:
C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\proyecto_ia_tumores_ovaricos\notebooks\R4_segmentation

Raíz del proyecto:
C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\proyecto_ia_tumores_ovaricos

Dataset:
C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\proyecto_ia_tumores_ovaricos\data\processed\MMOTU_OTU_2D

Verificación de rutas:
Train images         OK              C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\proyecto_ia_tumores_ovaricos\data\processed\MMOTU_OTU_2D\train\images
Train masks          OK              C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\proyecto_ia_tumores_ovaricos\data\processed\MMOTU_OTU_2D\train\masks
Validation images    OK              C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\proyecto_ia_tumores_ovaricos\data\processed\MMOTU_OTU_2D\validation\images
Validation masks     OK              C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\proyecto_ia_tumores_ovaricos\data\processed\MMOTU_OTU_2D\validation\masks
Metadata             OK     

In [4]:
def analyze_image_channels(image_path: Path) -> dict:
    """
    Analiza los canales de una imagen sin modificar el archivo original.

    Para imágenes RGB/RGBA calcula:
    - igualdad exacta entre R, G y B
    - diferencia máxima entre canales
    - diferencia media entre canales
    - proporción de píxeles donde existe alguna diferencia
    """

    with Image.open(image_path) as img:
        mode = img.mode
        array = np.asarray(img)

    result = {
        "image_name": image_path.name,
        "image_path": str(image_path),
        "mode": mode,
        "height": array.shape[0],
        "width": array.shape[1],
        "n_channels": 1 if array.ndim == 2 else array.shape[2],
        "rgb_exactly_equal": None,
        "max_channel_difference": None,
        "mean_channel_difference": None,
        "different_pixel_fraction": None,
    }

    # Imagen monocanal
    if array.ndim == 2:
        return result

    # Imagen con menos de 3 canales
    if array.shape[2] < 3:
        return result

    # Solo analizamos RGB.
    # Si existe canal alfa (RGBA), no interviene en esta comparación.
    rgb = array[:, :, :3].astype(np.int32)

    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]

    diff_rg = np.abs(r - g)
    diff_rb = np.abs(r - b)
    diff_gb = np.abs(g - b)

    pixel_has_difference = (
        (diff_rg > 0)
        | (diff_rb > 0)
        | (diff_gb > 0)
    )

    result["rgb_exactly_equal"] = bool(
        np.all(r == g) and np.all(g == b)
    )

    result["max_channel_difference"] = int(
        max(
            diff_rg.max(),
            diff_rb.max(),
            diff_gb.max()
        )
    )

    result["mean_channel_difference"] = float(
        np.mean(
            [
                diff_rg.mean(),
                diff_rb.mean(),
                diff_gb.mean()
            ]
        )
    )

    result["different_pixel_fraction"] = float(
        pixel_has_difference.mean()
    )

    return result

In [5]:
def get_image_files(directory: Path) -> list[Path]:

    if not directory.exists():
        raise FileNotFoundError(
            f"No se encontró el directorio: {directory.resolve()}"
        )

    files = [
        path
        for path in directory.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    ]

    return sorted(files)


train_files = get_image_files(TRAIN_IMAGES_DIR)
val_files = get_image_files(VAL_IMAGES_DIR)

print("Imágenes train:", len(train_files))
print("Imágenes validation:", len(val_files))
print("Total:", len(train_files) + len(val_files))

Imágenes train: 1000
Imágenes validation: 469
Total: 1469


In [6]:
records = []

for split, files in [
    ("train", train_files),
    ("validation", val_files),
]:
    for image_path in files:

        result = analyze_image_channels(image_path)
        result["split"] = split

        records.append(result)


channels_df = pd.DataFrame(records)

channels_df.head()

,image_name,image_path,mode,height,width,n_channels,rgb_exactly_equal,max_channel_difference,mean_channel_difference,different_pixel_fraction,split
0,1.JPG,C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\...,RGB,537,959,3,False,184,2.655128,0.770385,train
1,10.JPG,C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\...,RGB,542,483,3,False,174,0.443245,0.040514,train
2,100.JPG,C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\...,RGB,531,958,3,False,122,0.125610,0.014120,train
3,1000.JPG,C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\...,RGB,532,959,3,False,161,1.923464,0.770528,train
4,1001.JPG,C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\...,RGB,530,480,3,False,174,3.150537,0.913774,train


In [7]:
rgb_images = channels_df[
    channels_df["n_channels"] >= 3
].copy()

print("Imágenes con al menos 3 canales:", len(rgb_images))

if len(rgb_images) > 0:

    exact_equal = rgb_images["rgb_exactly_equal"].sum()

    print(
        f"R = G = B exactamente: "
        f"{exact_equal}/{len(rgb_images)} "
        f"({100 * exact_equal / len(rgb_images):.2f}%)"
    )

    print(
        "Máxima diferencia encontrada entre canales:",
        rgb_images["max_channel_difference"].max()
    )

    print(
        "Diferencia media global entre canales:",
        rgb_images["mean_channel_difference"].mean()
    )

    print(
        "Fracción media de píxeles diferentes:",
        rgb_images["different_pixel_fraction"].mean()
    )

Imágenes con al menos 3 canales: 1469
R = G = B exactamente: 1/1469 (0.07%)
Máxima diferencia encontrada entre canales: 255
Diferencia media global entre canales: 1.859373981812595
Fracción media de píxeles diferentes: 0.4098178726505774


In [8]:
different_images = rgb_images[
    rgb_images["rgb_exactly_equal"] == False
].copy()

print(
    "Imágenes con alguna diferencia entre R, G y B:",
    len(different_images)
)

display(
    different_images[
        [
            "split",
            "image_name",
            "mode",
            "max_channel_difference",
            "mean_channel_difference",
            "different_pixel_fraction",
        ]
    ]
    .sort_values(
        "max_channel_difference",
        ascending=False
    )
    .head(20)
)

Imágenes con alguna diferencia entre R, G y B: 1468


,split,image_name,mode,max_channel_difference,mean_channel_difference,different_pixel_fraction
904,train,866.JPG,RGB,255,5.597302,0.589295
887,train,845.JPG,RGB,255,4.678057,0.618579
912,train,876.JPG,RGB,255,3.877853,0.894244
911,train,875.JPG,RGB,255,2.088280,0.123157
910,train,874.JPG,RGB,255,1.032794,0.112495
909,train,873.JPG,RGB,255,1.582088,0.088788
895,train,853.JPG,RGB,255,0.968286,0.105438
67,train,1093.JPG,RGB,255,1.390491,0.104151
68,train,1094.JPG,RGB,255,0.608640,0.016896
893,train,851.JPG,RGB,255,6.701199,0.696693


In [9]:
OUTPUT_DIR = Path("reports/segmentation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUT_DIR / "input_channel_analysis.csv"

channels_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Reporte guardado en:")
print(output_file.resolve())

Reporte guardado en:
C:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\proyecto_ia_tumores_ovaricos\notebooks\R4_segmentation\reports\segmentation\input_channel_analysis.csv


In [10]:
import sys

print(sys.executable)

c:\Users\PC\Desktop\JARITZA - PUCP\26-2\Tesis\proyecto_ia_tumores_ovaricos\.venv\Scripts\python.exe


In [11]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cpu
CUDA disponible: False


In [12]:
import numpy as np
import torch

from PIL import Image
from torch.utils.data import Dataset, DataLoader


# CONFIGURACIÓN BASELINE

IMAGE_SIZE = (256, 256)

BATCH_SIZE = 4

IN_CHANNELS = 3
OUT_CHANNELS = 1

SEED = 42

In [13]:
import random

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cpu
CUDA disponible: False


In [19]:
def get_files_by_id(
    directory: Path,
    is_mask: bool = False,
    extensions=IMAGE_EXTENSIONS
):
    """
    Crea un índice:
        image_id -> Path

    Convención del dataset:
        imagen:   1000.JPG
        máscara:  1000_binary.PNG

    Para las máscaras elimina el sufijo '_binary'
    únicamente para obtener el ID de emparejamiento.
    """

    files = {}

    for path in directory.iterdir():

        if not path.is_file():
            continue

        if path.suffix.lower() not in extensions:
            continue

        file_id = path.stem

        if is_mask:
            suffix = "_binary"

            if not file_id.endswith(suffix):
                raise ValueError(
                    f"Máscara con nombre inesperado: {path.name}"
                )

            file_id = file_id[:-len(suffix)]

        # Evitar sobrescribir archivos accidentalmente
        if file_id in files:
            raise ValueError(
                f"ID duplicado '{file_id}' en {directory}"
            )

        files[file_id] = path

    return files

In [20]:
train_images = get_files_by_id(
    TRAIN_IMAGES_DIR,
    is_mask=False
)

train_masks = get_files_by_id(
    TRAIN_MASKS_DIR,
    is_mask=True
)

val_images = get_files_by_id(
    VAL_IMAGES_DIR,
    is_mask=False
)

val_masks = get_files_by_id(
    VAL_MASKS_DIR,
    is_mask=True
)

In [21]:
train_missing_masks = set(train_images) - set(train_masks)
train_missing_images = set(train_masks) - set(train_images)

val_missing_masks = set(val_images) - set(val_masks)
val_missing_images = set(val_masks) - set(val_images)


print("TRAIN")
print("Imágenes:", len(train_images))
print("Máscaras:", len(train_masks))
print("Imágenes sin máscara:", len(train_missing_masks))
print("Máscaras sin imagen:", len(train_missing_images))

print("\nVALIDATION")
print("Imágenes:", len(val_images))
print("Máscaras:", len(val_masks))
print("Imágenes sin máscara:", len(val_missing_masks))
print("Máscaras sin imagen:", len(val_missing_images))

TRAIN
Imágenes: 1000
Máscaras: 1000
Imágenes sin máscara: 0
Máscaras sin imagen: 0

VALIDATION
Imágenes: 469
Máscaras: 469
Imágenes sin máscara: 0
Máscaras sin imagen: 0


In [22]:
for image_id in list(sorted(train_images.keys(), key=int))[:10]:
    print(
        image_id,
        "->",
        train_images[image_id].name,
        "<->",
        train_masks[image_id].name
    )

1 -> 1.JPG <-> 1_binary.PNG
2 -> 2.JPG <-> 2_binary.PNG
4 -> 4.JPG <-> 4_binary.PNG
6 -> 6.JPG <-> 6_binary.PNG
7 -> 7.JPG <-> 7_binary.PNG
9 -> 9.JPG <-> 9_binary.PNG
10 -> 10.JPG <-> 10_binary.PNG
11 -> 11.JPG <-> 11_binary.PNG
13 -> 13.JPG <-> 13_binary.PNG
14 -> 14.JPG <-> 14_binary.PNG


In [23]:
# Codificacion de las mascaras
def inspect_mask_values(mask_paths):
    unique_values = set()

    for mask_path in mask_paths:
        with Image.open(mask_path) as mask:
            mask_array = np.asarray(mask)

        unique_values.update(
            np.unique(mask_array).tolist()
        )

    return sorted(unique_values)


train_mask_values = inspect_mask_values(
    list(train_masks.values())
)

val_mask_values = inspect_mask_values(
    list(val_masks.values())
)


print("Valores encontrados en máscaras TRAIN:")
print(train_mask_values)

print("\nValores encontrados en máscaras VALIDATION:")
print(val_mask_values)

Valores encontrados en máscaras TRAIN:
[0, 1]

Valores encontrados en máscaras VALIDATION:
[0, 1]


In [24]:
# Dataset definitivo
class OvarianTumorSegmentationDataset(Dataset):
    """
    Dataset para segmentación binaria de tumores ováricos.

    Imagen:
        RGB -> [3, H, W]
        float32 en [0, 1]

    Máscara:
        Binaria -> [1, H, W]
        float32 con valores {0, 1}
    """

    def __init__(
        self,
        images_dir: Path,
        masks_dir: Path,
        image_size=(256, 256),
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.image_size = image_size

        # ----------------------------------------------------
        # Indexación y emparejamiento
        # ----------------------------------------------------

        self.images = get_files_by_id(
            self.images_dir,
            is_mask=False
        )

        self.masks = get_files_by_id(
            self.masks_dir,
            is_mask=True
        )

        image_ids = set(self.images.keys())
        mask_ids = set(self.masks.keys())

        missing_masks = image_ids - mask_ids
        missing_images = mask_ids - image_ids

        if missing_masks:
            raise ValueError(
                f"{len(missing_masks)} imágenes no tienen máscara."
            )

        if missing_images:
            raise ValueError(
                f"{len(missing_images)} máscaras no tienen imagen."
            )

        # Orden numérico: 1, 2, 3...
        self.ids = sorted(
            image_ids,
            key=int
        )

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):

        image_id = self.ids[index]

        image_path = self.images[image_id]
        mask_path = self.masks[image_id]

        # ====================================================
        # IMAGEN
        # ====================================================

        with Image.open(image_path) as image:

            # Baseline: conservar entrada RGB original
            image = image.convert("RGB")

            image = image.resize(
                self.image_size,
                resample=Image.Resampling.BILINEAR
            )

            image = np.asarray(
                image,
                dtype=np.float32
            )

        # [0,255] -> [0,1]
        image = image / 255.0

        # HWC -> CHW
        image = np.transpose(
            image,
            (2, 0, 1)
        )

        # Copia contigua para PyTorch
        image = np.ascontiguousarray(image)

        image = torch.from_numpy(image)


        # ====================================================
        # MÁSCARA
        # ====================================================

        with Image.open(mask_path) as mask:

            mask = mask.convert("L")

            # NEAREST para preservar las etiquetas 0/1
            mask = mask.resize(
                self.image_size,
                resample=Image.Resampling.NEAREST
            )

            mask = np.asarray(
                mask,
                dtype=np.float32
            )

        # R2 ya verificó que las máscaras son binarias {0,1}.
        # Añadimos dimensión del canal:
        # [H,W] -> [1,H,W]
        mask = np.expand_dims(
            mask,
            axis=0
        )

        mask = np.ascontiguousarray(mask)

        mask = torch.from_numpy(mask)


        return {
            "image": image,
            "mask": mask,
            "image_id": image_id,
        }

In [25]:
# Crear datasets de entrenamiento y validación
IMAGE_SIZE = (256, 256)

train_dataset = OvarianTumorSegmentationDataset(
    images_dir=TRAIN_IMAGES_DIR,
    masks_dir=TRAIN_MASKS_DIR,
    image_size=IMAGE_SIZE,
)

val_dataset = OvarianTumorSegmentationDataset(
    images_dir=VAL_IMAGES_DIR,
    masks_dir=VAL_MASKS_DIR,
    image_size=IMAGE_SIZE,
)


print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

Train: 1000
Validation: 469


In [26]:
# Comprobar muestra
sample = train_dataset[0]

image = sample["image"]
mask = sample["mask"]

print("ID:", sample["image_id"])

print("\nIMAGEN")
print("Shape:", image.shape)
print("Dtype:", image.dtype)
print("Min:", image.min().item())
print("Max:", image.max().item())

print("\nMÁSCARA")
print("Shape:", mask.shape)
print("Dtype:", mask.dtype)
print("Valores:", torch.unique(mask))

ID: 1

IMAGEN
Shape: torch.Size([3, 256, 256])
Dtype: torch.float32
Min: 0.0
Max: 0.9882352948188782

MÁSCARA
Shape: torch.Size([1, 256, 256])
Dtype: torch.float32
Valores: tensor([0., 1.])


In [27]:
# DataLoaders
BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

In [28]:
# Primer batch real
batch = next(iter(train_loader))

images = batch["image"]
masks = batch["mask"]
image_ids = batch["image_id"]

print("Images:", images.shape)
print("Masks:", masks.shape)

print("\nImages dtype:", images.dtype)
print("Masks dtype:", masks.dtype)

print("\nImage range:")
print(
    images.min().item(),
    images.max().item()
)

print("\nMask values:")
print(torch.unique(masks))

print("\nIDs:")
print(image_ids)

Images: torch.Size([4, 3, 256, 256])
Masks: torch.Size([4, 1, 256, 256])

Images dtype: torch.float32
Masks dtype: torch.float32

Image range:
0.0 1.0

Mask values:
tensor([0., 1.])

IDs:
['119', '723', '1135', '1303']
